### Limpeza dos Metadados do Jogo (game_metadata.csv)
Este notebook tem um único objetivo: ler o arquivo game_metadata.csv, que contém informações sobre cada jogo, e extrair os dados de forma limpa.

O problema principal é que colunas como homeTeam e awayTeam estão salvas como strings que representam dicionários. Vamos parsear essas strings para extrair os IDs e criar um arquivo de mapa limpo (team_map.parquet) com as seguintes colunas:

- `game_id`
- `home_team_id`
- `away_team_id`

Este arquivo será a nossa fonte para identificar os times nos próximos scripts.

In [2]:
import pandas as pd
import ast # Usado para converter strings em dicionários de forma segura
import os

In [3]:
# Carregar o CSV Bruto
# Assumindo que o notebook e o csv estão na mesma pasta raiz do projeto
caminho_csv = 'game_metadata.csv' 

if not os.path.exists(caminho_csv):
    print(f"Erro: Arquivo '{caminho_csv}' não encontrado.")
    print("Por favor, coloque o game_metadata.csv na pasta raiz do projeto.")
else:
    df_raw = pd.read_csv(caminho_csv)
    print("Arquivo game_metadata.csv carregado com sucesso.")
    print("Visualizando o formato original das colunas problemáticas:")
    display(df_raw[['id', 'homeTeam', 'awayTeam']].head(2))

Arquivo game_metadata.csv carregado com sucesso.
Visualizando o formato original das colunas problemáticas:


,id,homeTeam,awayTeam
0,13335,"{'id': '5', 'name': 'Burnley', 'shortName': 'B...","{'id': '11', 'name': 'Manchester City', 'short..."
1,13336,"{'id': '2', 'name': 'Arsenal', 'shortName': 'A...","{'id': '221', 'name': 'Nottingham Forest', 'sh..."


In [4]:
# Função para Parsear as Strings e Limpeza dos Dados

def parse_dict_string(dict_string):
    """
    Converte uma string que se parece com um dicionário em um dicionário Python real.
    Usa ast.literal_eval por ser mais seguro que eval().
    """
    try:
        # Verifica se o valor é uma string antes de tentar parsear
        if isinstance(dict_string, str):
            return ast.literal_eval(dict_string)
        return None # Retorna None se não for uma string (ex: NaN)
    except (ValueError, SyntaxError):
        # Retorna None se a string não for um dicionário válido
        return None

print("Iniciando o parsing das colunas 'homeTeam' e 'awayTeam'...")

# 1. Aplica a função para converter as strings em dicionários
df_raw['homeTeam_dict'] = df_raw['homeTeam'].apply(parse_dict_string)
df_raw['awayTeam_dict'] = df_raw['awayTeam'].apply(parse_dict_string)

# 2. Extrai o 'id' de cada dicionário. O .str.get() lida com segurança com valores nulos.
df_raw['home_team_id'] = df_raw['homeTeam_dict'].str.get('id')
df_raw['away_team_id'] = df_raw['awayTeam_dict'].str.get('id')

# 3. Renomeia a coluna 'id' original para 'game_id' para maior clareza
df_raw.rename(columns={'id': 'game_id'}, inplace=True)

# 4. Cria o DataFrame final limpo apenas com as colunas que nos interessam
df_team_map = df_raw[['game_id', 'home_team_id', 'away_team_id', 'homeTeamStartLeft']].copy()

# 5. Converte as colunas para tipo numérico e remove linhas com erro
for col in df_team_map.columns:
    df_team_map[col] = pd.to_numeric(df_team_map[col], errors='coerce')
df_team_map.dropna(inplace=True)

# 6. Converte para inteiro para um formato mais limpo
for col in df_team_map.columns:
    df_team_map[col] = df_team_map[col].astype(int)

print("\nLimpeza concluída! Visualizando o DataFrame final:")
display(df_team_map.head())

Iniciando o parsing das colunas 'homeTeam' e 'awayTeam'...

Limpeza concluída! Visualizando o DataFrame final:


,game_id,home_team_id,away_team_id,homeTeamStartLeft
0,13335,5,11,1
1,13336,2,221,1
2,13337,1,19,0
3,13338,4,218,1
4,13339,8,54,0


In [5]:
# Salvar o Mapa de Times Limpo
caminho_saida = 'team_map.parquet'
df_team_map.to_parquet(caminho_saida, index=False)

print(f"Mapa de times limpo foi salvo com sucesso em: '{caminho_saida}'")

Mapa de times limpo foi salvo com sucesso em: 'team_map.parquet'
